# Wikimedia Attention and Hype Decay

This notebook uses real Wikimedia Analytics API pageviews to measure public attention cycles. Pageviews are attention, not truth or importance.

No synthetic fallback is used.


In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

# Prefer the checkout when this notebook is run inside the repository.
repo_root = Path.cwd()
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
    if (candidate / "src" / "detime").exists():
        repo_root = candidate
        break
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))

from examples.hot_trends.data import (
    HotTrendDataError,
    append_real_snapshot,
    build_arxiv_monthly_counts,
    fetch_coingecko_market_chart,
    fetch_defillama_stablecoin_chains,
    fetch_github_repo_metadata,
    fetch_github_stargazers,
    fetch_huggingface_models,
    fetch_wikipedia_pageviews,
    source_audit_table,
)
from examples.hot_trends.decomposition import (
    component_summary,
    decompose_table,
    editorial_priority,
    residual_event_table,
)
from examples.hot_trends.scoring import article_language_guardrails

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

CACHE_DIR = repo_root / "examples" / "hot_trends" / "cache"
OUTPUT_DIR = repo_root / "examples" / "hot_trends" / "outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_table(df, name):
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"saved: {path.relative_to(repo_root)}")
    return path


## 1. Select pages and date window


In [ ]:
articles = [
    "Artificial intelligence",
    "Large language model",
    "ChatGPT",
    "Bitcoin",
    "Nvidia",
    "OpenAI",
]
START = "2025-01-01"
END = "2026-05-20"
pd.DataFrame({"article": articles})


## 2. Fetch real pageviews


In [ ]:
frames = []
for article in articles:
    frames.append(fetch_wikipedia_pageviews(article, start=START, end=END))
views = pd.concat(frames, ignore_index=True)
views.head(20)


## 3. Audit the real pageview table


In [ ]:
audit = source_audit_table(views, value_col="views", entity_col="article", time_col="date")
audit


## 4. Decompose pageview attention


In [ ]:
components = decompose_table(views, entity_col="article", time_col="date", value_col="views", method="MA_BASELINE", period=7, trend_window=21, transform="log1p")
summary = editorial_priority(component_summary(components, entity_col="article", time_col="date"), entity_col="article")
summary


## 5. Residual shock events


In [ ]:
events = residual_event_table(components, entity_col="article", time_col="date", top_n=25)
events


## 6. Hype-decay table

A simple decay proxy: after each article's largest residual event, count days until residual drops below half of that peak.


In [ ]:
decay_rows = []
for article, sub in components.groupby("article"):
    sub = sub.sort_values("date").copy()
    rz = (sub["residual"] - sub["residual"].median()).abs()
    peak_idx = int(rz.idxmax())
    peak_date = pd.to_datetime(sub.loc[peak_idx, "date"])
    peak = float(rz.loc[peak_idx])
    after = sub.loc[peak_idx:].copy()
    after_rz = (after["residual"] - sub["residual"].median()).abs()
    below = after.loc[after_rz <= 0.5 * peak]
    half_life_days = None if below.empty else int((pd.to_datetime(below["date"].iloc[0]) - peak_date).days)
    decay_rows.append({"article": article, "peak_date": str(peak_date.date()), "peak_residual_abs": peak, "attention_half_life_days": half_life_days})
decay = pd.DataFrame(decay_rows).sort_values("peak_residual_abs", ascending=False)
decay


In [ ]:
save_table(audit, "05_wikipedia_attention_audit")
save_table(summary, "05_wikipedia_attention_summary")
save_table(events, "05_wikipedia_attention_events")
save_table(decay, "05_wikipedia_hype_decay")
save_table(article_language_guardrails(), "05_wikipedia_guardrails")
